# 2. Diagnóstico e limpeza

> ### ⚠️ Dados de exemplo são fictícios
> A base usada na demonstração foi gerada pelo notebook 1 e **não contém informação de pessoa ou empresa real**. As regras de diagnóstico e limpeza, no entanto, são reais e funcionam sobre qualquer CSV.
>
> Ao usar este notebook com dados verdadeiros, lembre que informação pessoal é protegida pela LGPD. Não publique a base nem o relatório em repositório público.

Lê qualquer arquivo CSV, aponta os problemas de qualidade, aplica as correções e entrega dois arquivos.

**Saídas:**
- `dados_tratados.csv` com a base corrigida
- `relatorio_qualidade.html` com o que foi encontrado e o que foi feito

Nenhuma alteração acontece em silêncio. Toda correção aplicada fica registrada no relatório.

In [ ]:
import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd

ARQUIVO = "cadastro_clientes.csv"     # troque aqui para analisar outro arquivo

SAIDA_DADOS = "dados_tratados.csv"
SAIDA_RELATORIO = "relatorio_qualidade.html"

problemas = []    # o que foi encontrado
acoes = []        # o que foi feito

def anotar_problema(coluna, titulo, qtd, total, detalhe, gravidade="medio"):
    problemas.append({"coluna": coluna, "titulo": titulo, "qtd": int(qtd),
                      "pct": qtd / total * 100 if total else 0,
                      "detalhe": detalhe, "gravidade": gravidade})

def anotar_acao(coluna, titulo, qtd, detalhe):
    acoes.append({"coluna": coluna, "titulo": titulo, "qtd": int(qtd), "detalhe": detalhe})

## Leitura

O separador e a codificação não são informados. A combinação que produzir mais colunas é a correta, porque separador errado faz a linha inteira virar uma coluna só.

In [ ]:
def ler(caminho):
    melhor = None
    for enc in ("utf-8", "latin-1"):
        for sep in (";", ",", "\t", "|"):
            try:
                d = pd.read_csv(caminho, sep=sep, encoding=enc, dtype=str)
            except Exception:
                continue
            if melhor is None or d.shape[1] > melhor[0].shape[1]:
                melhor = (d, sep, enc)
    if melhor is None:
        raise SystemExit("Nao foi possivel ler o arquivo.")
    return melhor


bruto, sep, enc = ler(ARQUIVO)
total = len(bruto)
print(f"{total} linhas x {len(bruto.columns)} colunas  ·  separador '{sep}'  ·  codificacao {enc}")
bruto.head(3)

## Funções de apoio

Duas conversões que resolvem os erros mais caros de tratamento de dados.

In [ ]:
def chave(t):
    """Reduz um texto a sua forma comparavel, para achar variacoes da mesma coisa."""
    t = unicodedata.normalize("NFKD", str(t))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", t.lower())


def para_numero(v):
    """
    Converte texto em numero aceitando padrao brasileiro e americano.

    A regra e posicional: havendo ponto e virgula juntos, o ponto e
    separador de milhar. Apagar todo ponto sem verificar transformaria
    89.90 em 8990, um erro de cem vezes que nao gera mensagem nenhuma.
    """
    s = str(v).strip().replace("R$", "").replace(" ", "")
    if not s:
        return None
    if "." in s and "," in s:
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def para_data(v):
    """
    Testa formatos explicitos em vez de deixar o pandas adivinhar.

    A adivinhacao le 03/05/2025 como 5 de marco pelo padrao americano,
    e o erro vai direto para a analise mensal sem aviso.
    """
    s = str(v).strip()
    for f in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%y", "%d.%m.%Y"):
        try:
            return pd.to_datetime(s, format=f)
        except (ValueError, TypeError):
            continue
    return pd.NaT


def eh_texto(serie):
    """
    Diz se a coluna ainda e textual.

    Testar dtype == object nao serve: versoes recentes do pandas usam um
    tipo proprio para texto, e a comparacao daria falso, fazendo o
    tratamento ser pulado sem nenhum aviso.
    """
    return not (pd.api.types.is_numeric_dtype(serie)
                or pd.api.types.is_datetime64_any_dtype(serie))


def eh_identificador(nome):
    """Identificador e rotulo, nao quantidade. Nao deve virar numero."""
    return bool(re.search(r"(^|_)(id|codigo|cod|matricula|cpf|cnpj|telefone|cep)(_|$)",
                          nome.lower()))


def eh_numerica(serie):
    amostra = serie[serie != ""].head(300)
    return len(amostra) > 0 and amostra.map(lambda v: para_numero(v) is not None).mean() > 0.85


def eh_data(serie):
    amostra = serie[serie != ""].head(300)
    return len(amostra) > 0 and amostra.map(lambda v: not pd.isna(para_data(v))).mean() > 0.85

## Diagnóstico e limpeza, coluna por coluna

Cada verificação registra o problema encontrado e a correção aplicada.

In [ ]:
df = bruto.fillna("").astype(str)

# --- espacos sobrando -------------------------------------------------
for c in df.columns:
    limpa = df[c].str.strip().str.replace(r"\s+", " ", regex=True)
    n = (df[c] != limpa).sum()
    if n:
        anotar_problema(c, "Espaços no início, no fim ou repetidos", n, total,
                        "Espaços invisíveis fazem o mesmo valor ser tratado como diferente.", "leve")
        anotar_acao(c, "Espaços removidos", n, "Texto normalizado com um único espaço entre palavras.")
    df[c] = limpa

# --- campos em branco -------------------------------------------------
for c in df.columns:
    n = (df[c] == "").sum()
    if n:
        grav = "grave" if n / total > 0.5 else "medio"
        anotar_problema(c, "Campos em branco", n, total,
                        "Coluna praticamente vazia." if n / total > 0.9
                        else "A coluna não está preenchida em parte dos registros.", grav)

print(f"{len(problemas)} problemas ate aqui")

In [ ]:
# --- numeros guardados como texto ------------------------------------
for c in df.columns:
    if eh_identificador(c) or not eh_numerica(df[c]):
        continue

    formatos = set()
    for v in df[c][df[c] != ""].head(300):
        if "R$" in v:
            formatos.add("com símbolo de moeda")
        elif "," in v and "." in v:
            formatos.add("ponto de milhar e vírgula decimal")
        elif "," in v:
            formatos.add("vírgula decimal")
        elif "." in v:
            formatos.add("ponto decimal")
        else:
            formatos.add("inteiro")

    preenchidos = (df[c] != "").sum()
    if len(formatos) > 1:
        anotar_problema(c, "Número escrito de várias formas", preenchidos, total,
                        "Formatos encontrados: " + ", ".join(sorted(formatos)) +
                        ". Somar sem tratar gera erro silencioso.", "grave")

    df[c] = df[c].map(para_numero)
    anotar_acao(c, "Convertido para número", preenchidos,
                "Separador decimal interpretado pela posição e símbolo de moeda removido.")

    # valores negativos onde nao deveria haver
    if any(p in c.lower() for p in ("total", "valor", "limite", "preco", "quantidade")):
        neg = int((df[c] < 0).sum())
        if neg:
            anotar_problema(c, "Valores negativos", neg, total,
                            "Valor negativo neste campo costuma indicar erro de lançamento.", "grave")
            df.loc[df[c] < 0, c] = None
            anotar_acao(c, "Negativos removidos", neg,
                        "Marcados como ausentes para não contaminar somas e médias.")

print("numeros tratados")

In [ ]:
# --- datas ------------------------------------------------------------
PADROES = [(r"^\d{2}/\d{2}/\d{4}$", "dia/mês/ano"),
           (r"^\d{4}-\d{2}-\d{2}$", "ano-mês-dia"),
           (r"^\d{2}-\d{2}-\d{2}$", "dia-mês-ano curto"),
           (r"^\d{2}\.\d{2}\.\d{4}$", "dia.mês.ano")]

for c in df.columns:
    if not eh_texto(df[c]) or not eh_data(df[c]):
        continue

    achados = {rot for pad, rot in PADROES
               for v in df[c][df[c] != ""].head(300) if re.match(pad, v)}
    preenchidos = (df[c] != "").sum()
    if len(achados) > 1:
        anotar_problema(c, "Datas em formatos diferentes", preenchidos, total,
                        "Formatos encontrados: " + ", ".join(sorted(achados)) +
                        ". Datas como 03/05 ficam ambíguas entre dia e mês.", "grave")

    df[c] = df[c].map(para_data)
    anotar_acao(c, "Convertida para data", preenchidos,
                "Formatos testados explicitamente, com prioridade para o padrão brasileiro.")

print("datas tratadas")

In [ ]:
# --- categorias escritas de varias formas -----------------------------
for c in df.columns:
    if not eh_texto(df[c]):
        continue
    preenchidos = df[c][df[c] != ""]
    if preenchidos.empty or preenchidos.nunique() > max(60, len(preenchidos) * 0.2):
        continue

    grupos = {}
    for v in preenchidos.unique():
        grupos.setdefault(chave(v), []).append(v)
    variacoes = {k: v for k, v in grupos.items() if len(v) > 1}
    if not variacoes:
        continue

    # a grafia mais frequente vira o padrao do grupo
    contagem = preenchidos.value_counts()
    padrao = {}
    for k, valores in grupos.items():
        padrao[k] = max(valores, key=lambda x: contagem.get(x, 0))

    afetados = int(preenchidos.map(lambda x: len(grupos[chave(x)]) > 1).sum())
    exemplos = " · ".join(" / ".join(v[:4]) for v in list(variacoes.values())[:2])

    anotar_problema(c, "Mesma categoria escrita de formas diferentes", afetados, total,
                    f"{len(variacoes)} grupos de valores equivalentes. Exemplos: {exemplos}. "
                    "Qualquer contagem agrupada por esta coluna sai fragmentada.", "grave")

    df[c] = df[c].map(lambda x: padrao[chave(x)] if x != "" else "")
    anotar_acao(c, "Categorias unificadas", afetados,
                "Cada grupo passou a usar a grafia mais frequente como padrão.")

print("categorias tratadas")

In [ ]:
# --- campos de sim ou nao ---------------------------------------------
SIM = {"s", "sim", "1", "true", "y", "yes", "v", "verdadeiro"}
NAO = {"n", "nao", "0", "false", "no", "f", "falso"}

for c in df.columns:
    if not eh_texto(df[c]):
        continue
    valores = {v.lower() for v in df[c].unique() if v != ""}
    if not valores or not valores <= (SIM | NAO):
        continue

    anotar_problema(c, "Sim e não escritos de várias formas", int((df[c] != "").sum()), total,
                    "Valores encontrados: " + ", ".join(sorted(valores)) +
                    ". Filtrar por esta coluna exige lembrar de todas as grafias.", "medio")
    df[c] = df[c].map(lambda v: "" if v == "" else ("Sim" if v.lower() in SIM else "Nao"))
    anotar_acao(c, "Padronizado para Sim e Não", int((df[c] != "").sum()),
                "Todas as grafias equivalentes passaram a usar a mesma forma.")

print("campos de sim ou nao tratados")

In [ ]:
# --- valores parecidos que exigem confirmacao -------------------------
# Abreviacoes nao sao corrigidas automaticamente. Juntar "Sta. Maria" com
# "Santa Maria" por semelhanca parece obvio, mas o mesmo criterio juntaria
# "Santo Angelo" com "Santa Angela". Correcao que adivinha e pior que
# problema declarado, entao aqui apenas sinalizamos.

for c in df.columns:
    if not eh_texto(df[c]):
        continue
    valores = [v for v in df[c].unique() if v != ""]
    if not 1 < len(valores) <= 40:
        continue

    pares = []
    for i, a in enumerate(valores):
        for b in valores[i + 1:]:
            ka, kb = chave(a), chave(b)
            if not ka or not kb:
                continue
            parecido = (SequenceMatcher(None, ka, kb).ratio() > 0.62
                        or ka.startswith(kb[:4]) or kb.startswith(ka[:4]))
            if parecido:
                pares.append(f"{a} / {b}")

    if pares:
        afetados = int(df[c].isin([v for p in pares for v in p.split(" / ")]).sum())
        anotar_problema(c, "Valores parecidos que podem ser a mesma coisa", afetados, total,
                        "Exemplos: " + " · ".join(pares[:3]) +
                        ". Não foram unificados automaticamente porque a semelhança pode ser "
                        "coincidência. Confirme com quem opera o sistema antes de corrigir.", "medio")

print("valores parecidos sinalizados")

In [ ]:
# --- documentos e email ----------------------------------------------
for c in df.columns:
    if not eh_texto(df[c]):
        continue
    nome = c.lower()

    if any(p in nome for p in ("cpf", "cnpj", "documento")):
        esperado = 11 if "cpf" in nome else 14
        digitos = df[c].map(lambda v: re.sub(r"\D", "", v))
        fora = int(((digitos.str.len() != esperado) & (df[c] != "")).sum())
        if fora:
            anotar_problema(c, "Documento com quantidade de dígitos incorreta", fora, total,
                            f"Esperado {esperado} dígitos. Fora disso não há como validar.", "grave")
        df[c] = digitos.where(digitos.str.len() == esperado, "")
        anotar_acao(c, "Documento padronizado", int((df[c] != "").sum()),
                    "Mantidos apenas os dígitos. Registros inválidos ficaram em branco.")

    if "mail" in nome:
        df[c] = df[c].str.lower()
        invalidos = int(((~df[c].str.contains("@", regex=False)) & (df[c] != "")).sum())
        if invalidos:
            anotar_problema(c, "Endereço de e-mail inválido", invalidos, total,
                            "Registros sem arroba não permitem contato.", "grave")
            df.loc[(~df[c].str.contains("@", regex=False)) & (df[c] != ""), c] = ""
            anotar_acao(c, "E-mails inválidos removidos", invalidos,
                        "Convertidos para branco e sinalizados para conferência manual.")

print("documentos tratados")

In [ ]:
# --- duplicatas -------------------------------------------------------
antes = len(df)

iguais = int(df.duplicated().sum())
if iguais:
    anotar_problema("tabela", "Linhas idênticas", iguais, total,
                    "Registros repetidos em todas as colunas, típicos de exportação dobrada.", "grave")
    df = df.drop_duplicates()
    anotar_acao("tabela", "Linhas idênticas removidas", iguais, "Mantida a primeira ocorrência.")

# mesma pessoa com identificador diferente
cols = [c for c in df.columns if not re.search(r"(^|_)(id|codigo|cod)(_|$)", c.lower())]
if cols and len(cols) < len(df.columns):
    rep = int(df.duplicated(subset=cols).sum())
    if rep:
        anotar_problema("tabela", "Cadastros repetidos com identificador diferente", rep, total,
                        "Mesma pessoa cadastrada mais de uma vez. Infla a contagem de clientes "
                        "e divide o histórico de compras.", "grave")
        df = df.drop_duplicates(subset=cols)
        anotar_acao("tabela", "Cadastros duplicados removidos", rep,
                    "Mantido o primeiro registro de cada pessoa.")

df = df.reset_index(drop=True)
print(f"{antes} linhas antes  ->  {len(df)} linhas depois")

## Nota de qualidade

Dentro de cada coluna considera-se o pior problema, não a soma. Os mesmos registros costumam ser atingidos por vários defeitos ao mesmo tempo, e somar contaria a mesma linha repetidas vezes, levando qualquer base minimamente suja a zero.

In [ ]:
PESOS = {"leve": 0.6, "medio": 1.0, "grave": 1.6}

por_coluna = {}
for p in problemas:
    if p["coluna"] == "tabela":
        continue
    peso = p["pct"] * PESOS[p["gravidade"]]
    por_coluna[p["coluna"]] = max(por_coluna.get(p["coluna"], 0), min(100, peso))

for c in bruto.columns:
    por_coluna.setdefault(c, 0)

media = sum(por_coluna.values()) / len(por_coluna)
tab = min(100, sum(p["pct"] * PESOS[p["gravidade"]] for p in problemas if p["coluna"] == "tabela"))
nota = max(0, round(100 - (media * 0.67 + tab * 0.33)))

rotulo = ("Base confiável" if nota >= 80
          else "Precisa de tratamento" if nota >= 60
          else "Base comprometida")

print(f"Nota {nota} de 100  ·  {rotulo}")
print(f"{len(problemas)} problemas encontrados  ·  {len(acoes)} correcoes aplicadas")

## Relatório

In [ ]:
GRAV = {"grave": ("Crítico", "#DC2626", "rgba(220,38,38,.08)"),
        "medio": ("Atenção", "#B45309", "rgba(180,83,9,.08)"),
        "leve":  ("Leve", "#2563EB", "rgba(37,99,235,.07)")}

cor_nota = "#0F172A" if nota >= 80 else "#B45309" if nota >= 60 else "#DC2626"

leitura = ("A base pode ser usada em análise com ajustes pontuais."
           if nota >= 80 else
           "A base precisa de tratamento antes de sustentar qualquer indicador."
           if nota >= 60 else
           "A base compromete qualquer análise feita sobre ela sem correção prévia.")

linhas_prob = "".join(
    f'<tr><td class="mono">{p["coluna"]}</td>'
    f'<td><span class="sit" style="color:{GRAV[p["gravidade"]][1]};'
    f'background:{GRAV[p["gravidade"]][2]}">{GRAV[p["gravidade"]][0]}</span></td>'
    f'<td><b>{p["titulo"]}</b><span class="det">{p["detalhe"]}</span></td>'
    f'<td class="n">{p["qtd"]}</td><td class="n muted">{p["pct"]:.1f}%</td></tr>'
    for p in sorted(problemas, key=lambda x: (-{"grave": 2, "medio": 1, "leve": 0}[x["gravidade"]], -x["pct"])))

linhas_acao = "".join(
    f'<tr><td class="mono">{a["coluna"]}</td>'
    f'<td><b>{a["titulo"]}</b><span class="det">{a["detalhe"]}</span></td>'
    f'<td class="n">{a["qtd"]}</td></tr>'
    for a in acoes)

graves = sum(1 for p in problemas if p["gravidade"] == "grave")
removidas = total - len(df)

html = f"""<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Diagnóstico de qualidade · {ARQUIVO}</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter+Tight:wght@400;500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
:root{{
  --bg:#FAFAFA; --ink:#0F172A; --primary:#2563EB; --surface:#FFFFFF;
  --muted:#5B6678; --faint:#8A93A3; --gray:#CBD5E1;
  --line:rgba(15,23,42,.09); --line-soft:rgba(15,23,42,.055); --tint:rgba(37,99,235,.07);
  --display:"Inter Tight","Inter",system-ui,sans-serif;
  --body:"Inter",system-ui,sans-serif;
  --mono:"JetBrains Mono",ui-monospace,Menlo,monospace;
  --r-md:14px; --r-lg:20px;
  --shadow-1:0 1px 2px rgba(15,23,42,.04), 0 1px 1px rgba(15,23,42,.03);
  --shadow-2:0 12px 32px -14px rgba(15,23,42,.16), 0 2px 6px rgba(15,23,42,.04);
}}
*,*::before,*::after{{box-sizing:border-box}}
body{{margin:0;background:var(--bg);color:var(--ink);font-family:var(--body);
  font-size:15px;line-height:1.6;letter-spacing:-.006em;-webkit-font-smoothing:antialiased}}
body::before{{content:"";position:fixed;inset:0;z-index:0;pointer-events:none;opacity:.5;
  background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='180' height='180'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.85' numOctaves='3'/%3E%3C/filter%3E%3Crect width='180' height='180' filter='url(%23n)' opacity='0.035'/%3E%3C/svg%3E")}}
.wrap{{max-width:980px;margin-inline:auto;padding:64px 24px 80px;position:relative;z-index:1}}

.eyebrow{{font-family:var(--mono);font-size:11px;font-weight:500;letter-spacing:.16em;
  text-transform:uppercase;color:var(--faint);display:flex;align-items:center;gap:10px;margin-bottom:18px}}
.eyebrow::before{{content:"";width:16px;height:1px;background:var(--gray)}}
h1{{font-family:var(--display);font-weight:600;letter-spacing:-.038em;line-height:1.05;
  font-size:clamp(30px,5vw,44px);margin:0 0 14px}}
.lead{{color:var(--muted);font-size:17px;max-width:60ch;margin:0}}
.meta{{font-family:var(--mono);font-size:11px;letter-spacing:.1em;text-transform:uppercase;
  color:var(--faint);margin-top:22px;display:flex;gap:14px;flex-wrap:wrap;align-items:center}}
.meta i{{width:12px;height:1px;background:var(--gray);display:block;font-style:normal}}

h2{{font-family:var(--display);font-weight:600;font-size:22px;letter-spacing:-.028em;
  margin:56px 0 6px}}
.sub{{color:var(--muted);font-size:14.5px;margin:0 0 20px;max-width:66ch}}

.nota{{margin-top:44px;background:var(--surface);border:1px solid var(--line);
  border-radius:var(--r-lg);box-shadow:var(--shadow-2);padding:32px;
  display:flex;gap:30px;align-items:center;flex-wrap:wrap}}
.nota__n{{font-family:var(--display);font-size:76px;font-weight:600;letter-spacing:-.05em;
  line-height:.9;color:{cor_nota}}}
.nota__n small{{font-size:19px;color:var(--faint);font-weight:500;letter-spacing:-.02em}}
.nota__txt{{flex:1;min-width:280px}}
.nota__txt b{{font-family:var(--display);font-size:19px;font-weight:600;letter-spacing:-.02em;display:block}}
.nota__txt p{{color:var(--muted);font-size:14.5px;margin:6px 0 0;max-width:52ch}}

.stats{{display:grid;grid-template-columns:repeat(4,1fr);border-top:1px solid var(--line);
  margin-top:34px}}
.stats div{{padding:20px 18px 20px 0;border-bottom:1px solid var(--line-soft)}}
.stats b{{display:block;font-family:var(--display);font-size:27px;font-weight:600;letter-spacing:-.032em}}
.stats span{{font-family:var(--mono);font-size:10.5px;letter-spacing:.11em;
  text-transform:uppercase;color:var(--faint)}}

.painel{{background:var(--surface);border:1px solid var(--line);border-radius:var(--r-lg);
  box-shadow:var(--shadow-1);overflow:hidden}}
table{{width:100%;border-collapse:collapse}}
th{{text-align:left;font-family:var(--mono);font-size:10px;font-weight:500;letter-spacing:.12em;
  text-transform:uppercase;color:var(--faint);padding:16px 18px 12px;
  border-bottom:1px solid var(--line);white-space:nowrap}}
td{{padding:15px 18px;border-bottom:1px solid var(--line-soft);vertical-align:top;font-size:14px}}
tr:last-child td{{border-bottom:0}}
td b{{font-family:var(--display);font-weight:600;letter-spacing:-.015em;display:block}}
td .det{{display:block;color:var(--muted);font-size:13px;margin-top:3px;max-width:62ch;line-height:1.55}}
td.n{{text-align:right;font-family:var(--mono);font-size:13px;white-space:nowrap}}
td.muted{{color:var(--faint)}}
td.mono{{font-family:var(--mono);font-size:12px;color:var(--muted);white-space:nowrap}}
th.r{{text-align:right}}
.sit{{font-family:var(--mono);font-size:10px;font-weight:500;letter-spacing:.07em;
  text-transform:uppercase;padding:4px 9px;border-radius:99px;white-space:nowrap}}

.ficticio{{margin-top:26px;border:1px solid rgba(180,83,9,.22);border-radius:var(--r-md);
  background:rgba(180,83,9,.055);padding:18px 20px;display:flex;gap:13px;align-items:flex-start}}
.ficticio svg{{width:17px;height:17px;color:#B45309;flex:none;margin-top:2px}}
.ficticio b{{font-family:var(--display);font-size:14px;font-weight:600;display:block;margin-bottom:3px;color:#7C3D06}}
.ficticio p{{margin:0;color:#8A5A22;font-size:13px;max-width:74ch;line-height:1.55}}

.aviso{{margin-top:56px;border:1px solid var(--line);border-radius:var(--r-md);
  background:var(--tint);padding:20px 22px;display:flex;gap:14px;align-items:flex-start}}
.aviso svg{{width:17px;height:17px;color:var(--primary);flex:none;margin-top:3px}}
.aviso b{{font-family:var(--display);font-size:14.5px;font-weight:600;display:block;margin-bottom:3px}}
.aviso p{{margin:0;color:var(--muted);font-size:13.5px;max-width:72ch}}

footer{{margin-top:56px;padding-top:24px;border-top:1px solid var(--line);
  display:flex;justify-content:space-between;gap:16px;flex-wrap:wrap;
  font-family:var(--mono);font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--faint)}}
footer b{{color:var(--ink);font-weight:500}}
@media(max-width:720px){{
  .wrap{{padding:44px 18px 60px}}
  .stats{{grid-template-columns:1fr 1fr}}
  .nota{{padding:24px}}
  td,th{{padding-inline:14px}}
}}
</style></head><body><div class="wrap">

<p class="eyebrow">Diagnóstico de qualidade de dados</p>
<h1>O que impede esta base<br>de sustentar uma análise</h1>
<p class="lead">Leitura automática do arquivo recebido, apontando os problemas que
distorcem indicadores e registrando cada correção aplicada.</p>
<p class="meta"><span>{ARQUIVO}</span><i></i><span>separador &ldquo;{sep}&rdquo;</span>
<i></i><span>codificação {enc}</span></p>

<div class="ficticio">
  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8"
    stroke-linecap="round"><path d="M12 9v4M12 17h.01M10.3 3.9 1.8 18a2 2 0 0 0 1.7 3h17a2 2 0 0 0 1.7-3L13.7 3.9a2 2 0 0 0-3.4 0z"/></svg>
  <div><b>Demonstração com dados fictícios</b>
  <p>Os registros analisados neste relatório foram gerados por script para fins de portfólio.
  Nomes, documentos, contatos e valores são sorteados e não correspondem a nenhuma pessoa ou
  empresa real. Os CPFs não possuem dígito verificador válido. A metodologia de diagnóstico e
  limpeza é real e se aplica a qualquer base.</p></div>
</div>

<div class="nota">
  <div class="nota__n">{nota}<small>/100</small></div>
  <div class="nota__txt"><b>{rotulo}</b><p>{leitura}</p></div>
</div>

<div class="stats">
  <div><b>{total}</b><span>linhas recebidas</span></div>
  <div><b>{len(df)}</b><span>linhas entregues</span></div>
  <div><b>{len(problemas)}</b><span>problemas encontrados</span></div>
  <div><b>{len(acoes)}</b><span>correções aplicadas</span></div>
</div>

<h2>Problemas encontrados</h2>
<p class="sub">Ordenados por gravidade. A coluna de percentual indica a fatia dos registros
atingida por cada problema.</p>
<div class="painel"><table>
<thead><tr><th>Coluna</th><th>Gravidade</th><th>Problema</th>
<th class="r">Registros</th><th class="r">Participação</th></tr></thead>
<tbody>{linhas_prob}</tbody></table></div>

<h2>Correções aplicadas</h2>
<p class="sub">Nenhum registro foi alterado sem constar nesta lista. É o que permite
reconciliar os números tratados com o sistema de origem.</p>
<div class="painel"><table>
<thead><tr><th>Coluna</th><th>Ação</th><th class="r">Registros</th></tr></thead>
<tbody>{linhas_acao}</tbody></table></div>

<div class="aviso">
  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8"
    stroke-linecap="round"><circle cx="12" cy="12" r="10"/><path d="M12 16v-4M12 8h.01"/></svg>
  <div><b>Como ler a nota</b>
  <p>A nota parte de 100 e desconta a proporção de registros comprometidos, ponderada pela
  gravidade. Dentro de cada coluna considera-se o pior problema, e não a soma, porque os
  mesmos registros costumam ser atingidos por mais de um defeito ao mesmo tempo. Somar
  contaria a mesma linha várias vezes e levaria qualquer base minimamente suja a zero.</p></div>
</div>

<div class="aviso">
  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8"
    stroke-linecap="round"><path d="M12 9v4M12 17h.01M10.3 3.9 1.8 18a2 2 0 0 0 1.7 3h17a2 2 0 0 0 1.7-3L13.7 3.9a2 2 0 0 0-3.4 0z"/></svg>
  <div><b>O que não foi corrigido automaticamente</b>
  <p>Valores apenas parecidos, como abreviações, ficam sinalizados mas intactos. Unificar
  por semelhança juntaria corretamente &ldquo;Sta. Maria&rdquo; e &ldquo;Santa Maria&rdquo;, mas o mesmo critério
  juntaria &ldquo;Santo Ângelo&rdquo; e &ldquo;Santa Ângela&rdquo;. Correção que adivinha é pior do que problema
  declarado, então a decisão fica com quem conhece a operação.</p></div>
</div>

<footer>
  <span>{graves} problemas críticos · {removidas} linhas removidas por duplicidade</span>
  <span>Dados fictícios · Relatório gerado por <b>MV Engenharia de Dados</b></span>
</footer>
</div></body></html>"""

Path(SAIDA_RELATORIO).write_text(html, encoding="utf-8")
print(f"{SAIDA_RELATORIO} gravado")

## Saída dos dados tratados

In [ ]:
df.to_csv(SAIDA_DADOS, index=False, encoding="utf-8")
print(f"{SAIDA_DADOS} gravado com {len(df)} linhas e {len(df.columns)} colunas")
df.head(5)

## Baixar os arquivos

A célula abaixo funciona no Google Colab. Fora dele, os arquivos ficam na mesma pasta do notebook.

In [ ]:
try:
    from google.colab import files
    files.download(SAIDA_DADOS)
    files.download(SAIDA_RELATORIO)
except ImportError:
    print("Fora do Colab. Os arquivos estao na pasta do notebook.")